# Inference Configuration
Specify
- path

In [2]:
import pandas as pd

csv_path = "../../output/two-model/v1/test_predicted.csv"

data = pd.read_csv(csv_path)
data

,domain_id,domain_start,domain_end,class,architecture,topology,homology,protein_sequence,real_domain_start,real_domain_end,cath,protein_length,protein_id,protein_chain_id,cath_prediction
0,1r4gA00,1,61,1,10,8,10,KPTMHSLRLVIESSPLSRAEKAAYVKSLSKCKTDQEVKAVMELVEE...,1,53,1.10.8.10,53,1r4g,1r4gA,1.20.58.90
1,1a5tA02,183,324,1,10,8,10,MRWYPWLRPDFEKLVASYQAGRGHHALLIQALPGMGDDALIYALSR...,168,207,1.10.8.10,334,1a5t,1a5tA,1.20.272.10
2,2damA00,4,69,1,10,8,10,GSSGSSGAPEERDLTQEQTEKLLQFQDLTGIESMDQCRHTLEQHNW...,1,67,1.10.8.10,67,2dam,2damA,1.10.8.10
3,1r5lA01,1,268,1,10,8,20,GSHMSPLLQPGLAALRRRAREAGVPLAPLPLTDSFLLRFLRARDFD...,9,64,1.10.8.20,262,1r5l,1r5lA,3.40.525.10
4,3twlA02,4,277,1,10,8,50,MPELPEVEAARRAIEENCLGKKIKRVIIADDNKVIHGISPSDFQTS...,146,287,1.10.8.50,310,3twl,3twlA,1.10.8.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1394,1rl0A02,1,259,4,10,470,10,ATAYTLNLANPSASQYSSFLDQIRNNVRDTSLIYGGTDVAVIGAPS...,182,255,4.10.470.10,255,1rl0,1rl0A,3.40.420.10
1395,1owfB00,1,102,4,10,520,10,MTKSELIERLATQQSHIPAKTVEDAVKEMLEHMASTLAQGERIAIR...,1,94,4.10.520.10,94,1owf,1owfB,4.10.520.10
1396,1e52A00,5,70,4,10,860,10,MHHHHHHLEPDNVPMDMSPKALQQKIHELEGLMMQHAQNLEFEEAA...,8,63,4.10.860.10,63,1e52,1e52A,1.10.287.130
1397,3n93B04,96,378,4,10,1240,10,MAKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLE...,409,476,4.10.1240.10,482,3n93,3n93B,3.40.47.10


In [3]:
import pickle

label_encoder_path = "../../output/protenn2/v5/label_encoder.pkl"
with open(label_encoder_path, "rb") as f:
    label_encoder = pickle.load(f)

In [4]:
import numpy as np

y_true_labels_list = []
y_pred_labels_list = []
protein_chain_id_list = []
no_domain_label_id = label_encoder.transform(["NO_DOMAIN_REGION"])[0]
for protein_chain_id, grp in data.groupby('protein_chain_id'):
    length = grp['protein_length'].iat[0]
    # use numpy arrays instead of Python lists
    y_true = np.full(length, no_domain_label_id, dtype=object)
    y_pred = np.full(length, no_domain_label_id, dtype=object)

    for _, row in grp.iterrows():
        pred_id = int(label_encoder.transform([row['cath_prediction']])[0])
        true_id = int(label_encoder.transform([row['cath']])[0])
        # predicted-domain fill
        ps, pe = int(row['domain_start']), int(row['domain_end'])
        y_pred[ps:pe] = pred_id

        # true-domain fill
        ts, te = int(row['real_domain_start']) - 1, int(row['real_domain_end'])
        y_true[ts:te] = true_id
    y_true_labels_list.append(y_true)
    y_pred_labels_list.append(y_pred)
    protein_chain_id_list.append(protein_chain_id)

In [5]:
y_pred_labels_list

[array([877, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241, 241,
        241, 241, 241, 241, 877, 877, 877, 877, 877, 877, 877, 877, 877,
        877, 877, 877, 877, 877, 877, 877, 877, 877, 877], dtype=object),
 array([877, 877, 877, 877, 877, 877, 877, 877, 87

In [6]:

from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

y_true_labels_list = [y_true.astype(int) for y_true in y_true_labels_list]
y_pred_labels_list = [y_pred.astype(int) for y_pred in y_pred_labels_list]

# then one-hot encode using your mapper’s class count
num_classes = mapper.get_class_count("H")

y_pred_confidences_list = [
    np.eye(num_classes, dtype=int)[y_pred]
    for y_pred in y_pred_labels_list]


In [7]:
import pickle
data_dict = {}
for true_labels, pred_labels, protein_chain_id in zip(y_true_labels_list, y_pred_labels_list, protein_chain_id_list):
    data_dict[protein_chain_id] = {"true_labels": true_labels, "pred_labels": pred_labels}


In [8]:
with open("../../output/protenn2/v5/data_bene_better_format.pkl", "wb") as f:
    pickle.dump(data_dict, f)

# Analysis Configuration

In [6]:


bootstrap_samples = 1000
post_process_kwargs = None
post_process_func = None
metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
                      "segment_overlap_score")

# metrics_to_compute = ("segment_overlap_score")

In [7]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=post_process_func,
                                                post_process_kwargs=post_process_kwargs,
                                                metrics_to_compute=metrics_to_compute)

---- Computing Metrics for hierarchy: C
----- Accuracy Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:02<00:00, 376.44it/s]


{'mean': 0.696722578528457, 'ci_lower': np.float64(0.676372012545652), 'ci_upper': np.float64(0.7152159948867678), 'alpha': 0.05}
----- F1 Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:13<00:00, 74.46it/s]


{'mean': 0.6759644428566425, 'ci_lower': np.float64(0.6544961541279105), 'ci_upper': np.float64(0.6967564222099747), 'alpha': 0.05}
----- Jaccard Score Metrics -----


...

{'mean': 0.7215480253189651, 'ci_lower': np.float64(0.6960446637537606), 'ci_upper': np.float64(0.74576247750652), 'alpha': 0.05}
----- Precision Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:41<00:00, 24.13it/s]


{'mean': 0.7206147704128621, 'ci_lower': np.float64(0.7361484996476008), 'ci_upper': np.float64(0.7805386341596253), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:55<00:00, 17.96it/s]

{'mean': 71.67967746575893, 'ci_lower': np.float64(69.12262456689692), 'ci_upper': np.float64(74.23224848608825), 'alpha': 0.05}


# All results

In [ ]:
all_results

In [8]:
import json

with open("../../output/protenn2/v5/metrics_test_bene.json", "w") as f:
    json.dump(all_results, f)

In [ ]:
plt = plot_metric_with_ci(all_results, "segment_overlap_score")
plt.show()

In [ ]:
post_process_kwargs["no_domain_label_id"] = test_dataset.no_domain_encoded_id